In [1]:
import sys
!{sys.executable} -m pip install -q pandas numpy joblib xgboost openpyxl huggingface_hub httpx
!{sys.executable} -m pip install -q torch
!{sys.executable} -m pip install -q esm

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


This cell installs the necessary Python libraries for the project. These include `pandas` for data manipulation, `numpy` for numerical operations, `joblib` for model serialization, `xgboost` for the machine learning model, `openpyxl` for Excel file handling, `huggingface_hub` for downloading models from Hugging Face, and `esm` for ESM-C protein embeddings.

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import joblib
import torch

from huggingface_hub import hf_hub_download

c:\Users\conra\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


This cell imports the Python libraries that will be used throughout the notebook.
- `os` and `pickle` are standard Python libraries.
- `numpy` and `pandas` are for numerical operations and data manipulation, respectively.
- `joblib` is used to load the pre-trained machine learning model and label encoder.
- `torch` is the underlying framework for the ESM-C model.
- `huggingface_hub` is used to download files from Hugging Face, and `google.colab.files` is for interacting with the Colab environment, specifically for file uploads and downloads.

In [3]:
# =========================
# USER SETTINGS
# =========================
HF_REPO_ID = "KarunaAnna/CritiCL"
MODEL_FILENAME = "model_XGB.joblib"          # required
LABEL_ENCODER_FILENAME = "label_encoder.joblib"  # optional; set to None if not available
HF_REPO_TYPE = "model"                       # "model" or "dataset"
HF_REVISION = None                           # e.g. "main" or a commit hash


This cell defines user-configurable settings.
- `HF_REPO_ID`: Specifies the Hugging Face repository where the model is stored.
- `MODEL_FILENAME`: The name of the main model file to be downloaded.
- `LABEL_ENCODER_FILENAME`: The name of the label encoder file, if available. It's set to `None` if no label encoder is used.
- `HF_REPO_TYPE`: Indicates whether the repository contains a 'model' or 'dataset'.
- `HF_REVISION`: Allows specifying a particular version or commit hash of the repository, or `None` for the latest version.

In [4]:
model_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=MODEL_FILENAME,
    repo_type=HF_REPO_TYPE,
    revision=HF_REVISION,
)

print("Downloaded model to:", model_path)

model = joblib.load(model_path)
print("Loaded model type:", type(model))
print("Model expects n_features =", model.n_features_in_)

Downloaded model to: C:\Users\conra\.cache\huggingface\hub\models--KarunaAnna--CritiCL\snapshots\1a389c328df8c66fb5c7c0eb8df668bff37cb76c\model_XGB.joblib
Loaded model type: <class 'xgboost.sklearn.XGBClassifier'>
Model expects n_features = 960


c:\Users\conra\AppData\Local\Python\pythoncore-3.14-64\Lib\pickle.py:1835: UserWarning: [21:30:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\gbm\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


This cell downloads the specified machine learning model from Hugging Face using `hf_hub_download` and then loads it using `joblib`. It then prints the path where the model was downloaded, its type, and the number of features it expects as input.

In [5]:
label_encoder = None

if LABEL_ENCODER_FILENAME is not None:
    try:
        label_encoder_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=LABEL_ENCODER_FILENAME,
            repo_type=HF_REPO_TYPE,
            revision=HF_REVISION,
        )
        label_encoder = joblib.load(label_encoder_path)
        print("Loaded label encoder classes:", list(label_encoder.classes_))
    except Exception as e:
        print("Could not load label encoder.")
        print("Reason:", e)

Loaded label encoder classes: ['Co', 'Ln', 'Mn', 'Ni', 'Zn']


This cell attempts to download and load a `label_encoder` if `LABEL_ENCODER_FILENAME` is set. A label encoder is used to convert numerical labels back into their original string representations. If loaded successfully, it prints the classes the encoder was trained on. It also includes error handling in case the label encoder cannot be loaded.

In [6]:
def load_esmc(model_name: str = "esmc_300m", device: str = None):
    from esm.models.esmc import ESMC

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = ESMC.from_pretrained(model_name).to(device)
    model.eval()
    return model, device


model_esmc, device = load_esmc(model_name="esmc_300m")
print("ESM-C loaded on:", device)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\conra\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\conra\.cache\huggingface\hub\models--EvolutionaryScale--esmc-300m-2024-12. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or

ESM-C loaded on: cpu


This cell defines and calls the `load_esmc` function, which loads the ESM-C (Evolutionary Scale Modeling - Context) model. ESM-C is a pre-trained protein language model that can generate embeddings (numerical representations) of protein sequences. The function checks for GPU availability (`cuda`) and loads the model onto the appropriate device (GPU if available, otherwise CPU). The model is set to evaluation mode (`model.eval()`).

In [7]:
@torch.no_grad()
def esmc_token_embeddings_aligned(model, sequence: str, device: str):
    """
    Mirrors your uploaded embedding script.
    Returns aligned per-residue embeddings of shape (L, d).
    """
    from esm.sdk.api import ESMProtein, LogitsConfig

    sequence = str(sequence).strip().upper()
    L = len(sequence)

    protein = ESMProtein(sequence=sequence)
    protein_tensor = model.encode(protein).to(device)

    out = model.logits(protein_tensor, LogitsConfig(sequence=True, return_embeddings=True))
    emb = out.embeddings

    if isinstance(emb, np.ndarray):
        emb = torch.from_numpy(emb)
    emb = emb.to(device).detach()

    if emb.dim() == 3:
        if emb.shape[0] != 1:
            raise RuntimeError(f"Unexpected batch dim: {tuple(emb.shape)}")
        emb = emb.squeeze(0)

    if emb.dim() != 2:
        raise RuntimeError(f"Expected 2D embeddings (T,d), got {tuple(emb.shape)}")

    T = emb.shape[0]
    if T == L:
        return emb
    if T == L + 2:
        return emb[1:-1, :]
    if T == L + 1:
        return emb[1:, :]
    if T > L:
        start = (T - L) // 2
        return emb[start:start + L, :]

    raise RuntimeError(f"Cannot align embeddings: tokens={T}, seq_len={L}, emb_shape={tuple(emb.shape)}")


def mean_pool_residue_embeddings(emb_Ld):
    return emb_Ld.mean(dim=0)


def clean_sequence(seq):
    if pd.isna(seq):
        return ""
    return str(seq).strip().replace(" ", "").replace("\n", "").upper()


@torch.no_grad()
def sequence_to_embedding(seq, model, device):
    seq = clean_sequence(seq)
    if not seq:
        raise ValueError("Empty sequence provided.")

    emb_Ld = esmc_token_embeddings_aligned(model, seq, device=device)
    emb_d = mean_pool_residue_embeddings(emb_Ld)

    emb = emb_d.detach().cpu().numpy().astype(np.float32, copy=False)
    return emb

This cell contains several helper functions crucial for processing protein sequences:
- `esmc_token_embeddings_aligned`: This function takes the ESM-C model, a protein sequence, and a device (CPU/GPU) as input. It generates per-residue embeddings from the ESM-C model and handles alignment to ensure the embeddings match the sequence length. This involves encoding the protein, obtaining logits (predictions), extracting embeddings, and handling various alignment scenarios based on the embedding dimensions.
- `mean_pool_residue_embeddings`: Takes the per-residue embeddings and performs mean pooling to get a single, fixed-size embedding for the entire protein sequence.
- `clean_sequence`: A utility function to clean up protein sequences by removing whitespace, newlines, and converting to uppercase, and handling NaN values.
- `sequence_to_embedding`: This is the main function that orchestrates the embedding generation. It cleans the input sequence, calls `esmc_token_embeddings_aligned` to get per-residue embeddings, and then `mean_pool_residue_embeddings` to get a single sequence embedding. It returns the embedding as a NumPy array.

In [8]:
test_seq = "AKWYFGLICCKLQLK"
test_emb = sequence_to_embedding(test_seq, model_esmc, device)

print("Generated embedding dim:", test_emb.shape[0])
print("Model expected dim:", model.n_features_in_)

if test_emb.shape[0] != model.n_features_in_:
    raise ValueError(
        f"Embedding dimension mismatch: generated {test_emb.shape[0]}, "
        f"but model expects {model.n_features_in_}."
    )

Generated embedding dim: 960
Model expected dim: 960


This cell serves as a test to ensure the embedding generation process works correctly. It takes a sample protein sequence (`test_seq`), generates its embedding using `sequence_to_embedding`, and then checks if the dimension of the generated embedding matches the number of features the pre-trained `model` (XGBoost classifier) expects. If there's a mismatch, it raises a `ValueError`.

In [9]:
def predict_from_dataframe(df, seq_col="Sequence", cycl_col="CyclizationPattern"):
    if seq_col not in df.columns:
        raise ValueError(f"Missing required sequence column: {seq_col}")

    work = df.copy()

    if cycl_col not in work.columns:
        work[cycl_col] = ""

    embeddings = []
    for seq in work[seq_col]:
        emb = sequence_to_embedding(seq, model_esmc, device)
        embeddings.append(emb)

    X = np.vstack(embeddings)

    if X.shape[1] != model.n_features_in_:
        raise ValueError(
            f"Feature shape mismatch, expected: {model.n_features_in_}, got {X.shape[1]}"
        )

    y_pred = model.predict(X)

    out = work.reset_index(drop=True).copy()

    if label_encoder is not None:
        try:
            out["prediction"] = label_encoder.inverse_transform(np.asarray(y_pred, dtype=int))
        except Exception:
            out["prediction"] = y_pred
    else:
        out["prediction"] = y_pred

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        out["confidence_max"] = proba.max(axis=1)

        if label_encoder is not None:
            class_names = list(label_encoder.classes_)
        elif hasattr(model, "classes_"):
            class_names = [str(c) for c in model.classes_]
        else:
            class_names = [f"class_{i}" for i in range(proba.shape[1])]

        for i, cname in enumerate(class_names):
            out[f"proba_{cname}"] = proba[:, i]

    return out

This cell defines the `predict_from_dataframe` function, which is central to making predictions.
- It takes a DataFrame containing protein sequences and an optional cyclization pattern column.
- For each sequence, it generates an embedding using `sequence_to_embedding`.
- These embeddings are then stacked into a NumPy array `X`.
- It performs a sanity check to ensure the feature dimension of `X` matches the model's expectation.
- The `model.predict(X)` is called to get predictions.
- If a `label_encoder` is available, predictions are inverse-transformed to their original class names.
- If the model supports `predict_proba`, it also calculates and adds prediction probabilities for each class and the maximum confidence to the output DataFrame. The function returns a new DataFrame with the original data plus prediction and confidence columns.

In [10]:
def run_single_sequence():
    seq = input("Enter sequence: ").strip()
    cyc = input("Enter cyclization pattern (blank allowed, metadata only): ").strip()

    df = pd.DataFrame([{
        "Sequence": seq,
        "CyclizationPattern": cyc
    }])

    result = predict_from_dataframe(df)
    return result

single_result = run_single_sequence()
single_result

,Sequence,CyclizationPattern,prediction,confidence_max,proba_Co,proba_Ln,proba_Mn,proba_Ni,proba_Zn
0,A,,Ni,0.555522,0.128522,0.193003,0.085185,0.555522,0.037768


This cell defines `run_single_sequence`, a function that prompts the user to enter a single protein sequence and an optional cyclization pattern. It then creates a Pandas DataFrame from this input, calls `predict_from_dataframe` to get the prediction, and returns the result. The cell then calls this function and displays the output for a single sequence.

In [11]:
def run_multiple_sequences():
    n = int(input("How many sequences do you want to enter? ").strip())
    rows = []

    for i in range(n):
        print(f"\nSequence {i+1}")
        seq = input("  Enter sequence: ").strip()
        cyc = input("  Enter cyclization pattern (blank allowed, metadata only): ").strip()

        rows.append({
            "Sequence": seq,
            "CyclizationPattern": cyc
        })

    df = pd.DataFrame(rows)
    result = predict_from_dataframe(df)
    return result

multi_result = run_multiple_sequences()
multi_result


Sequence 1

Sequence 2


,Sequence,CyclizationPattern,prediction,confidence_max,proba_Co,proba_Ln,proba_Mn,proba_Ni,proba_Zn
0,A,,Ni,0.555522,0.128522,0.193003,0.085185,0.555522,0.037768
1,C,,Ln,0.542083,0.133873,0.542083,0.073527,0.220618,0.029899


This cell defines `run_multiple_sequences`, allowing the user to input several sequences manually.
- It prompts the user for the number of sequences they want to enter.
- Then, in a loop, it asks for each sequence and its cyclization pattern.
- These inputs are collected into a DataFrame, and `predict_from_dataframe` is called to generate predictions for all sequences.
- The cell then executes this function and displays the results.
*(Note: The `KeyboardInterrupt` in the output indicates that the execution was stopped by the user during input for multiple sequences, not an error in the code itself.)*

This cell was an empty text cell.

In [12]:
def run_multiple_sequences():
    n = int(input("How many sequences do you want to enter? ").strip())
    rows = []

    for i in range(n):
        print(f"\nSequence {i+1}")
        seq = input("  Enter sequence: ").strip()
        cyc = input("  Enter cyclization pattern (blank allowed, metadata only): ").strip()

        rows.append({
            "Sequence": seq,
            "CyclizationPattern": cyc
        })

    df = pd.DataFrame(rows)
    result = predict_from_dataframe(df)
    return result

multi_result = run_multiple_sequences()
multi_result


Sequence 1


,Sequence,CyclizationPattern,prediction,confidence_max,proba_Co,proba_Ln,proba_Mn,proba_Ni,proba_Zn
0,A,,Ni,0.555522,0.128522,0.193003,0.085185,0.555522,0.037768


This cell is a duplicate of the previous `run_multiple_sequences` definition and execution. It appears there might have been an accidental re-execution or copy-paste of the same logic.
*(Note: Another `KeyboardInterrupt` occurred here, again indicating user interruption during input.)*

In [17]:
def run_uploaded_file():
    in_file = input("Enter path to CSV or Excel file: ").strip()

    if in_file.lower().endswith(".csv"):
        df = pd.read_csv(in_file, keep_default_na=False, na_values=[])
    elif in_file.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(in_file)
    else:
        raise ValueError("Please provide a CSV or Excel file path.")

    print("Detected columns:", list(df.columns))
    seq_col = input("Enter sequence column name [default: Sequence]: ").strip() or "Sequence"
    cycl_col = input("Enter cyclization column name [default: CyclizationPattern]: ").strip() or "CyclizationPattern"

    result = predict_from_dataframe(df, seq_col=seq_col, cycl_col=cycl_col)
    return result

upload_result = run_uploaded_file()
upload_result.head()

Detected columns: ['Sequence']


,Sequence,CyclizationPattern,prediction,confidence_max,proba_Co,proba_Ln,proba_Mn,proba_Ni,proba_Zn
0,AA,,Ni,0.556042,0.295222,0.076236,0.043590,0.556042,0.028910
1,AC,,Co,0.307569,0.307569,0.209915,0.086940,0.305933,0.089642
2,AD,,Ni,0.650134,0.211394,0.070112,0.035606,0.650134,0.032754
3,AE,,Ni,0.586044,0.310601,0.045768,0.031709,0.586044,0.025878
4,AF,,Co,0.726046,0.726046,0.039128,0.031067,0.160393,0.043366


As mentioned, `upload_result.head()` only shows the first 5 rows. To see all the results from your uploaded file, we can display the entire `upload_result` DataFrame:

In [15]:
display(upload_result)

,Sequence,CyclizationPattern,prediction,confidence_max,proba_Co,proba_Ln,proba_Mn,proba_Ni,proba_Zn
0,A,,Ni,0.555522,0.128522,0.193003,0.085185,0.555522,0.037768
1,C,,Ln,0.542083,0.133873,0.542083,0.073527,0.220618,0.029899
2,D,,Ni,0.581793,0.120531,0.154863,0.090250,0.581793,0.052563
3,E,,Ni,0.459875,0.226188,0.168768,0.067821,0.459875,0.077348
4,F,,Ln,0.402480,0.199486,0.402480,0.090821,0.271763,0.035450
5,G,,Ni,0.531837,0.157555,0.205227,0.067537,0.531837,0.037844
6,H,,Ni,0.535730,0.082339,0.287811,0.055921,0.535730,0.038199
7,I,,Ni,0.405577,0.121257,0.338192,0.089449,0.405577,0.045525
8,K,,Ni,0.448953,0.151485,0.131800,0.151995,0.448953,0.115768
9,L,,Ni,0.375372,0.144828,0.269311,0.115648,0.375372,0.094841


This cell defines `run_uploaded_file`, a function for making predictions using sequences from an uploaded CSV or Excel file.
- It uses `files.upload()` to allow the user to select a file.
- It then reads the file into a Pandas DataFrame, supporting both CSV and Excel formats.
- It prompts the user to confirm the column names for `Sequence` and `CyclizationPattern`, providing defaults.
- Finally, it calls `predict_from_dataframe` with the data from the uploaded file and displays the head of the results.


In [16]:
def save_results(df, default_name="xgb_predictions.csv"):
    out_name = input(f"Output filename [default: {default_name}]: ").strip() or default_name

    if out_name.lower().endswith(".csv"):
        df.to_csv(out_name, index=False)
    elif out_name.lower().endswith((".xlsx", ".xls")):
        df.to_excel(out_name, index=False)
    else:
        out_name += ".csv"
        df.to_csv(out_name, index=False)

    print("Saved to:", os.path.abspath(out_name))

# Example:
# save_results(single_result)
# save_results(multi_result)
save_results(upload_result)

Saved to: c:\Users\conra\Documents\code\Chowdhury\metal-binding\VSTEST.csv


This cell defines `save_results`, a utility function to save the prediction results to a file.
- It prompts the user for an output filename, providing a default.
- It automatically appends `.csv` or `.xlsx` based on the user's input or defaults to `.csv`.
- The results DataFrame is saved to the specified file, and then `files.download()` is used to initiate a download of the file in the Colab environment.
- Examples of how to call this function are commented out.

In [ ]:
def run_menu():
    print("\nChoose input mode:")
    print("1 = Single sequence")
    print("2 = Multiple sequences manually")
    print("3 = Upload CSV/Excel")

    choice = input("Enter 1, 2, or 3: ").strip()

    if choice == "1":
        res = run_single_sequence()
    elif choice == "2":
        res = run_multiple_sequences()
    elif choice == "3":
        res = run_uploaded_file()
    else:
        raise ValueError("Invalid choice. Please enter 1, 2, or 3.")

    display(res)
    return res

results_df = run_menu()

This cell defines `run_menu`, a function that provides a user interface to choose between different input methods: single sequence, multiple sequences (manual entry), or uploading a file.
- It prompts the user to select an option (1, 2, or 3).
- Based on the choice, it calls the corresponding `run_single_sequence`, `run_multiple_sequences`, or `run_uploaded_file` function.
- It then displays the results and returns the resulting DataFrame.
- The cell concludes by calling `run_menu()` to start the interactive prediction process.

Let's load the `sequences_2aa.csv` file and inspect its 'Sequence' column. We will apply the `clean_sequence` function (defined earlier) to each entry and identify if any result in an empty string, which would cause the error.

In [ ]:
df_2aa = pd.read_csv('/content/sequences_2aa.csv')

# Apply the clean_sequence function to the 'Sequence' column
df_2aa['cleaned_sequence'] = df_2aa['Sequence'].apply(clean_sequence)

# Find rows where the cleaned sequence is empty
empty_sequence_rows = df_2aa[df_2aa['cleaned_sequence'] == '']

if not empty_sequence_rows.empty:
    print("Found rows with empty sequences after cleaning:")
    display(empty_sequence_rows[['Sequence', 'cleaned_sequence']])
else:
    print("No empty sequences found after cleaning in sequences_2aa.csv.")


To clarify what pandas loaded, let's directly access and print the content of the 'Sequence' column at index 220 in the `df_2aa` DataFrame:

In [ ]:
print(f"Value in df_2aa['Sequence'] at index 220: {df_2aa['Sequence'].loc[220]}")
print(f"Type of value: {type(df_2aa['Sequence'].loc[220])}")

Let's reload `sequences_2aa.csv` with `keep_default_na=False` and `na_values=[]` to ensure pandas treats everything as a string and doesn't convert any values to `NaN` by default. Then we'll inspect the problematic cell at index 220 again, and see how our `clean_sequence` function handles it in this scenario.

In [ ]:
df_2aa_raw = pd.read_csv('/content/sequences_2aa.csv', keep_default_na=False, na_values=[''])

problem_cell_value = df_2aa_raw['Sequence'].loc[220]
cleaned_problem_cell = clean_sequence(problem_cell_value)

print(f"Value in 'Sequence' at index 220 (raw from CSV): '{problem_cell_value}'")
print(f"Type of value (raw from CSV): {type(problem_cell_value)}")
print(f"Value after clean_sequence: '{cleaned_problem_cell}'")
print(f"Length of cleaned value: {len(cleaned_problem_cell)}")

In [ ]:
predictions_2aa = predict_from_dataframe(df_2aa_raw)
display(predictions_2aa)